In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using backend: {device}")

Using backend: mps


In [5]:
import math

# ----------------------------------------------------------------------
# Scaling-law calculator (Chinchilla compute-optimal)
# Sources:
#   C ~= 6ND                  -> Kaplan 2020 (Sec 2.1) / Hoffmann 2022 (App. F)
#   D ~= 20 * N (tokens/param) -> Hoffmann 2022 (Chinchilla), Table 3
#   H100 BF16 dense throughput -> NVIDIA H100 datasheet (~989 TFLOP/s)
# ----------------------------------------------------------------------

# Hardware throughput constants (peak, dense, FLOP/s)
H100_BF16_DENSE = 989e12   # NVIDIA H100 SXM, BF16/FP16 Tensor Core, no sparsity
A100_BF16_DENSE = 312e12   # NVIDIA A100, BF16, for comparison
L4_BF16_DENSE = 121e12 


def compute_budget(num_gpus, hours, peak_flops_per_s, mfu=0.40):
    """
    Total training compute C, in FLOPs.
    mfu = Model FLOPs Utilization, the fraction of peak you actually sustain.
          ~0.40 is realistic for a well-tuned run; lower for a naive setup.
    """
    effective = peak_flops_per_s * mfu
    seconds = hours * 3600
    return num_gpus * effective * seconds


def optimal_split(C, tokens_per_param=20.0):
    """
    Chinchilla-optimal (N, D) for a fixed compute budget C.
    Solves C = 6 * N * D with the constraint D = tokens_per_param * N:
        C = 6 * N * (tokens_per_param * N) = 6 * tokens_per_param * N^2
        N = sqrt(C / (6 * tokens_per_param))
    Returns N (params), D (tokens).
    """
    N = math.sqrt(C / (6 * tokens_per_param))
    D = tokens_per_param * N
    return N, D


def report(num_gpus, hours, peak_flops_per_s, mfu=0.40, tokens_per_param=20.0):
    """Pretty-print a full budget -> optimal-model breakdown."""
    C = compute_budget(num_gpus, hours, peak_flops_per_s, mfu)
    N, D = optimal_split(C, tokens_per_param)
    print(f"Budget:      {num_gpus} GPU x {hours} h  @ {mfu:.0%} MFU")
    print(f"Compute C:   {C:.3e} FLOPs")
    print(f"Optimal N:   {N/1e6:,.1f} M params   ({N:.3e})")
    print(f"Optimal D:   {D/1e9:,.2f} B tokens   ({D:.3e})")
    print(f"Sanity 6ND:  {6*N*D:.3e}   (should match C)")
    print(f"Ratio D/N:   {D/N:.1f} tokens/param")

report(num_gpus=1, hours=1, peak_flops_per_s=L4_BF16_DENSE, mfu=0.30)

Budget:      1 GPU x 1 h  @ 30% MFU
Compute C:   1.307e+17 FLOPs
Optimal N:   33.0 M params   (3.300e+07)
Optimal D:   0.66 B tokens   (6.600e+08)
Sanity 6ND:  1.307e+17   (should match C)
Ratio D/N:   20.0 tokens/param


# Dataset

In [5]:
from torch.utils.data import Dataset
from torch import tensor

class CustomDataset(Dataset):
  def __init__(self, indices, context_size, stride):
    self.indices = indices

    self.x = []
    self.y = []
    for i in range(0, len(self.indices), stride):
      if (i + context_size + 1) > len(self.indices):
        break
      self.x.append(self.indices[i:i+context_size])
      self.y.append(self.indices[i+1:i+context_size+1])

    self.inputs = tensor(self.x)
    self.targets = tensor(self.y)

  def __len__(self):
      return len(self.inputs)

  def __getitem__(self, index):
    return self.inputs[index], self.targets[index]

# MODEL

In [6]:
import torch.nn.functional as F
import torch.nn as nn

class Multi_Head_Attention(nn.Module):
  def __init__(self, context_size, embedding_dim, proj_dim, n_heads):
    super().__init__()

    self.n_heads = n_heads
    if proj_dim % n_heads != 0:
      raise ValueError("proj_dim must be divisible by n_heads")
    self.head_dim = proj_dim // n_heads

    self.proj_dim = proj_dim
    self.query_layer = nn.Linear(embedding_dim, proj_dim, bias=False)
    self.key_layer = nn.Linear(embedding_dim, proj_dim, bias=False)
    self.value_layer = nn.Linear(embedding_dim, proj_dim, bias=False)

    self.output_proj = nn.Linear(proj_dim, embedding_dim, bias=False)

    self.register_buffer(
      "mask",
      torch.full((context_size, context_size),float("-inf")).triu(diagonal=1),
      persistent=False,
    )

  def forward(self, x):
    B, T, D = x.shape

    query_matrix = self.query_layer(x)
    key_matrix = self.key_layer(x)
    value_matrix = self.value_layer(x)

    # We have shape (B, T, D) for query, key, and value
    # Should reshape it to (B, T, H, HD) for all 3 and then to
    # (B, H, T, HD) and then matmul with key
    query_matrix = query_matrix.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    key_matrix = key_matrix.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    value_matrix = value_matrix.reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    dot_attention = torch.matmul(query_matrix, key_matrix.transpose(-2, -1)) + self.mask[:T, :T]
    # Result is of shape (B, H, T, T)

    scaled_dot_attention = dot_attention / (key_matrix.shape[-1] ** 0.5)
    softmax_output = F.softmax(scaled_dot_attention, dim=-1)

    # Shapes are (B, H, T, T) and (B, H, T, HD), result is (B, H, T, HD) before transpose
    final_attention = torch.matmul(softmax_output, value_matrix).transpose(1, 2)
    final_attention = final_attention.reshape(B, T, self.proj_dim)


    return self.output_proj(final_attention)

In [7]:
class FFN(nn.Module):
  def __init__(self, embedding_dim):
    super().__init__()

    self.input_proj = nn.Linear(embedding_dim, embedding_dim * 4, bias=False)
    self.output_proj = nn.Linear(embedding_dim * 4, embedding_dim, bias=False)

  def forward(self, x):
    return self.output_proj(F.relu(self.input_proj(x)))

In [8]:
class TransformerBlock(nn.Module):
  def __init__(self, context_size, embedding_dim, proj_dim, n_heads):
    super().__init__()

    self.mha = Multi_Head_Attention(context_size, embedding_dim, proj_dim, n_heads)
    self.layer_norm_1 = nn.LayerNorm(embedding_dim)
    self.ffn = FFN(embedding_dim=embedding_dim)
    self.layer_norm_2 = nn.LayerNorm(embedding_dim)

  def forward(self, x):
    x = x + self.mha(self.layer_norm_1(x))
    return x + self.ffn(self.layer_norm_2(x))


In [9]:
class GPT_Model(nn.Module):
  def __init__(self, vocab_size, context_size, embedding_dim, proj_dim, n_heads, n_layers):
    super().__init__()

    self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
    self.pos_embedding_layer = nn.Embedding(context_size, embedding_dim)

    self.transformer_blocks = nn.Sequential(
        *[TransformerBlock(context_size, embedding_dim, proj_dim, n_heads) for _ in range(n_layers)]
    )
    self.final_norm = nn.LayerNorm(embedding_dim)
    self.output_proj = nn.Linear(embedding_dim, vocab_size, bias=False)

  def forward(self, x):
    B, T = x.shape

    token_embeddings = self.embedding_layer(x)
    pos_embeddings = self.pos_embedding_layer(torch.arange(T).to(device))

    input_embeddings = token_embeddings + pos_embeddings

    transformer_output = self.transformer_blocks(input_embeddings)
    normalized = self.final_norm(transformer_output)
    logits = self.output_proj(normalized)

    return logits

In [10]:
def cross_entropy_loss(logits, targets):
  B, T, C = logits.shape
  probs = F.softmax(logits, dim=-1).flatten(start_dim=0, end_dim=1)
  # Currently probs has shape (B * T, C) and targets has shape (B*T)

  targets = targets.flatten(start_dim=0, end_dim=1)
  selected_probs = probs[torch.arange(B*T), targets]

  return -torch.log(selected_probs).mean()

In [11]:
def train(model, train_loader, val_loader, optimizer, epochs=5):
  step = 0
  history = {'epoch': [], 'train_loss': [], 'val_loss': []}

  for epoch in range(epochs):
    model.train()
    print(f"Started Epoch #{epoch}")
    train_loss_total = 0

    for batch in iter(train_loader):
      inputs, targets = batch[0].to(device), batch[1].to(device)

      logits = model(inputs)
      loss = cross_entropy_loss(logits, targets)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_loss_total += loss.item()
      step += 1
      if step % 100 == 0:
        print(f"Step {step} loss: {loss.item()}")

    avg_train_loss = train_loss_total / len(train_loader)

    # Validation Phase
    model.eval()
    val_loss_total = 0
    for batch in iter(val_loader):
      inputs, targets = batch[0].to(device), batch[1].to(device)

      with torch.no_grad():
        logits = model(inputs)
        loss = cross_entropy_loss(logits, targets)
        val_loss_total += loss.item()

    avg_val_loss = val_loss_total / len(val_loader)
    print(f"Epoch #{epoch} completed. Average Train Loss: {avg_train_loss:.4f}, Average Validation loss: {avg_val_loss:.4f}")

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

  return history

In [26]:
from torch.utils.data import DataLoader

config = {
    "vocab_size": len(tokenizer.idx_to_char),
    "context_size": 512,
    "embedding_dim": 128,
    "proj_dim": 256,
    "n_heads": 8,
    "n_layers": 8
}

train_set = CustomDataset(train_indices, context_size=512, stride=512)
val_set = CustomDataset(val_indices, context_size=512, stride=512)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

model = GPT_Model(**config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

history = train(model, train_loader, val_loader, optimizer, 10)

In [2]:
import matplotlib.pyplot as plt

plt.plot(history['epoch'], history['train_loss'], label='Training Loss')
plt.plot(history['epoch'], history['val_loss'], label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss Over Time')
plt.legend()
plt.show()

In [ ]:
import torch

def predict(model, text, tokenizer, output_length=50):
  encoded = tokenizer.encode(text, char_to_idx)

  context = torch.tensor(encoded).unsqueeze(0).to(device)

  model.eval()

  for _ in range(output_length):

    with torch.no_grad():
      logits = model(context)

    idx = torch.argmax(logits[0, -1]).unsqueeze(0).unsqueeze(0).to(device)

    context = torch.cat((context, idx), dim=1)

  return tokenizer.decode(context, idx_to_char)

new_text = predict(model, "Why ", tokenizer, 10)
print(new_text)

Why he came to
